# Downloading JRC GSW Yearly History — year 2000 (permanent + seasonal)

Source: [JRC/GSW1_4/YearlyHistory](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_YearlyHistory).

The `waterClass` band has values: 0 = no data, 1 = not water, 2 = seasonal water, 3 = permanent water.
This notebook exports two binary masks for year 2000: permanent (`waterClass == 3`) and seasonal (`waterClass == 2`).

Exports go to Google Drive via Earth Engine. Monitor progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import os
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize()

In [3]:
# Define export region (global)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

# Year of interest
YEAR = 2000

# Load the JRC Yearly History image for the chosen year.
yearly = ee.Image(f"JRC/GSW1_4/YearlyHistory/{YEAR}").select("waterClass")

# Binary masks: permanent (==3) and seasonal (==2) water
perm_2000     = yearly.eq(3)
seasonal_2000 = yearly.eq(2)

In [4]:
# Export permanent water (waterClass == 3) in YEAR
task = ee.batch.Export.image.toDrive(
    image=perm_2000,
    description=f'permwater_yh{YEAR}_100m_30m',
    folder='GEE_exports',
    fileNamePrefix=f'permwater_yh{YEAR}_100m_30m',
    region=world_bbox,
    scale=100,
    maxPixels=1e13
)
task.start()

In [5]:
# Export seasonal water (waterClass == 2) in YEAR
task = ee.batch.Export.image.toDrive(
    image=seasonal_2000,
    description=f'seasonalwater_yh{YEAR}_100m_30m',
    folder='GEE_exports',
    fileNamePrefix=f'seasonalwater_yh{YEAR}_100m_30m',
    region=world_bbox,
    scale=100,
    maxPixels=1e13
)
task.start()

### NOTE
- Track export progress at the [GEE Task Manager](https://code.earthengine.google.com/tasks).
- After downloads finish, place the tiles under `Measures_work/maps/raw/Water_surface/permwater_yh2000/` and `.../seasonalwater_yh2000/` so a downstream `*_ethnologue.ipynb` can glob them by folder.